# Getting Started with the Market Study Playground

**Welcome!** This notebook introduces the Market Study Playground - a space for exploring markets, testing ideas, and learning quantitative techniques without formal research requirements.

## What You'll Learn

1. How to load market data (equities, bonds, commodities, macro)
2. Basic data manipulation and visualization
3. Common analysis patterns (returns, volatility, correlation)
4. How to save your studies
5. Next steps for deeper exploration

## Key Difference from Research

- **Playground**: Fast exploration, no rigor requirements, learning-focused
- **Research**: Formal backtesting, statistical gates, PM review, production-ready

Let's get started!

## 1. Setup

In [11]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# Add project root to path
project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

# Import playground helpers
from workstation.playground.data_helpers import (
    get_prices,
    get_macro_series,
    get_market_snapshot,
    get_correlation_matrix,
    calculate_returns,
    calculate_volatility,
    calculate_drawdown,
    FRED_SERIES
)

print("Setup complete!")

Setup complete!


## 2. Loading Market Data

The playground provides simplified data access through `data_helpers.py`.

### Load Single Asset

In [12]:
# IAU — iShares Gold ETF price history
iau = get_prices('IAU', start='2005-01-01')

# Build Plotly figure with price + volume
fig = go.Figure()

# Price area chart
fig.add_trace(go.Scatter(
    x=iau['date'], y=iau['close'],
    mode='lines',
    name='IAU Close',
    line=dict(color='#D4AF37', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(212,175,55,0.10)',
))

# Key event annotations
events = [
    ('2008-09-15', 'GFC'),
    ('2011-09-06', 'Gold peak'),
    ('2020-03-23', 'COVID crash'),
    ('2020-08-07', '2020 ATH'),
    ('2024-10-30', '2024 ATH'),
]
for date_str, label in events:
    dt = pd.Timestamp(date_str)
    row = iau[iau['date'] >= dt]
    if row.empty:
        continue
    row = row.iloc[0]
    fig.add_annotation(
        x=row['date'], y=row['close'],
        text=label, showarrow=True,
        arrowhead=2, arrowcolor='#aaa', arrowwidth=1,
        font=dict(size=10, color='#888'),
        ay=-40, ax=0,
    )

fig.update_layout(
    title=dict(text='IAU — iShares Gold ETF (2005 – present)', font=dict(size=15)),
    xaxis=dict(title='Date', showgrid=True, gridcolor='rgba(200,200,200,0.2)'),
    yaxis=dict(title='Price (USD)', tickprefix='$', showgrid=True, gridcolor='rgba(200,200,200,0.2)'),
    height=500,
    template='plotly_dark',
    hovermode='x unified',
    showlegend=False,
)
fig.show()

# Summary stats
print(f"IAU Summary ({iau['date'].min().date()} – {iau['date'].max().date()})")
print(f"  First close:   ${iau['close'].iloc[0]:.2f}")
print(f"  Last close:    ${iau['close'].iloc[-1]:.2f}")
print(f"  All-time high: ${iau['close'].max():.2f}  on {iau.loc[iau['close'].idxmax(), 'date'].date()}")
print(f"  All-time low:  ${iau['close'].min():.2f}  on {iau.loc[iau['close'].idxmin(), 'date'].date()}")
print(f"  Total return:  {(iau['close'].iloc[-1] / iau['close'].iloc[0] - 1) * 100:.1f}%")

IAU Summary (2005-01-28 – 2026-03-23)
  First close:   $8.54
  Last close:    $82.78
  All-time high: $101.57  on 2026-01-29
  All-time low:  $8.26  on 2005-02-08
  Total return:  869.5%


### Load Multiple Assets

In [3]:
# Load multiple assets at once
tickers = ['SPY', 'TLT', 'GLD']
prices = get_prices(tickers, start='2020-01-01', end='2024-12-31')

# Result is a dictionary
for ticker, df in prices.items():
    print(f"{ticker}: {len(df)} rows")

SPY: 0 rows
TLT: 0 rows
GLD: 0 rows


### Load Macro Data (FRED)

In [4]:
# Load VIX from FRED
vix = get_macro_series('VIXCLS', start='2020-01-01')

print(f"Loaded {len(vix)} rows")
vix.head()

Loaded 0 rows


,date,series_id,value


## 3. Basic Visualization

### Price Chart

In [ ]:
# Simple price chart
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=spy['date'],
    y=spy['close'],
    name='SPY',
    line=dict(color='blue')
))
fig.update_layout(
    title='SPY Price History',
    xaxis_title='Date',
    yaxis_title='Price ($)',
    hovermode='x unified'
)
fig.show()

### Multiple Assets

In [ ]:
# Normalize to 100 for comparison
fig = go.Figure()

for ticker, df in prices.items():
    normalized = df['close'] / df['close'].iloc[0] * 100
    fig.add_trace(go.Scatter(
        x=df['date'],
        y=normalized,
        name=ticker
    ))

fig.update_layout(
    title='Normalized Performance (100 = start)',
    xaxis_title='Date',
    yaxis_title='Index',
    hovermode='x unified'
)
fig.show()

## 4. Common Analysis Patterns

### Calculate Returns

In [ ]:
# Simple returns
spy_returns = calculate_returns(spy['close'], method='simple')

print(f"Mean daily return: {spy_returns.mean():.4%}")
print(f"Std daily return: {spy_returns.std():.4%}")
print(f"Annualized return: {spy_returns.mean() * 252:.2%}")
print(f"Annualized vol: {spy_returns.std() * np.sqrt(252):.2%}")

### Calculate Volatility

In [ ]:
# Rolling volatility (20-day, annualized)
vol = calculate_volatility(spy_returns, window=20, annualize=True)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=spy['date'][20:],
    y=vol[20:] * 100,
    name='20-day Vol',
    line=dict(color='red')
))
fig.update_layout(
    title='SPY Rolling Volatility (20-day, annualized)',
    xaxis_title='Date',
    yaxis_title='Volatility (%)',
    hovermode='x unified'
)
fig.show()

### Correlation Analysis

In [ ]:
# Correlation matrix
corr = get_correlation_matrix(['SPY', 'TLT', 'GLD'], window=60)

# Plot heatmap
fig = px.imshow(
    corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1
)
fig.update_layout(title='Asset Correlation Matrix (60-day)')
fig.show()

## 5. Next Steps

Now that you know the basics, explore these resources:

### Get Help
- Ask **Tutor agent** for educational guidance
- Ask **Explorer agent** for study ideas

### Resources
- `playground/README.md` - Playground overview
- `playground/QUICK_REFERENCE.md` - Common tasks cheat sheet
- `playground/studies/TEMPLATE.md` - Study documentation template

Happy exploring!

In [5]:
import pandas as pd

# Check what tickers are in equities.parquet
eq = pd.read_parquet('/Users/zelin/Desktop/PA Investment/Invest_strategy/data/market_data/prices/equities.parquet')
print('equities.parquet tickers:', sorted(eq['ticker'].unique().tolist()))
print('Date range:', eq['date'].min(), 'to', eq['date'].max())
print()

# Check spy_ohlc.parquet
spy = pd.read_parquet('/Users/zelin/Desktop/PA Investment/Invest_strategy/data/market_data/prices/spy_ohlc.parquet')
print('spy_ohlc.parquet columns:', spy.columns.tolist())
print('spy_ohlc shape:', spy.shape)
print('Date range:', spy.index.min() if spy.index.dtype != 'object' else spy['date'].min(), 'to', spy.index.max() if spy.index.dtype != 'object' else spy['date'].max())
spy.head()

equities.parquet tickers: ['AAPL', '^GSPC', '^N225', '^NDX', '^RUT', '^STOXX', '^VIX']
Date range: 2024-02-26 to 2026-02-27

spy_ohlc.parquet columns: ['open', 'high', 'low', 'close', 'volume']
spy_ohlc shape: (5070, 5)
Date range: 2006-01-03 00:00:00 to 2026-02-27 00:00:00


,open,high,low,close,volume
date,,,,,
2006-01-03,125.190002,127.000000,124.389999,126.699997,73256700
2006-01-04,126.860001,127.489998,126.699997,127.300003,51899600
2006-01-05,127.150002,127.589996,126.879997,127.379997,47307500
2006-01-06,128.020004,128.580002,127.360001,128.440002,62885900
2006-01-09,128.419998,129.059998,128.380005,128.770004,43527400


In [7]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/zelin/Desktop/PA Investment/Invest_strategy/data/market_data/prices")

# Load existing equities.parquet
eq = pd.read_parquet(DATA_DIR / "equities.parquet")
print("Before merge — tickers:", sorted(eq['ticker'].unique()))
print("Shape:", eq.shape)

# Load spy_ohlc — date is the index
spy_raw = pd.read_parquet(DATA_DIR / "spy_ohlc.parquet")
spy = (
    spy_raw
    .reset_index()
    .rename(columns={"index": "date"})
    .assign(ticker="SPY")
)[["date", "ticker", "open", "high", "low", "close", "volume"]]

spy["date"] = pd.to_datetime(spy["date"])
eq["date"] = pd.to_datetime(eq["date"])

print(f"\nSPY rows to add: {len(spy)}, range: {spy['date'].min().date()} to {spy['date'].max().date()}")

combined = (
    pd.concat([eq, spy], ignore_index=True)
    .drop_duplicates(subset=["date", "ticker"])
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

print("\nAfter merge — tickers:", sorted(combined['ticker'].unique()))
print("Shape:", combined.shape)

combined.to_parquet(DATA_DIR / "equities.parquet", index=False)
print("\nSaved equities.parquet")

verify = pd.read_parquet(DATA_DIR / "equities.parquet")
spy_check = verify[verify['ticker'] == 'SPY']
print(f"SPY rows: {len(spy_check)}, range: {spy_check['date'].min().date()} to {spy_check['date'].max().date()}")

Before merge — tickers: ['AAPL', 'IAU', 'SPY', '^GSPC', '^N225', '^NDX', '^RUT', '^STOXX', '^VIX']
Shape: (13895, 7)

SPY rows to add: 5070, range: 2006-01-03 to 2026-02-27

After merge — tickers: ['AAPL', 'IAU', 'SPY', '^GSPC', '^N225', '^NDX', '^RUT', '^STOXX', '^VIX']
Shape: (13895, 7)

Saved equities.parquet
SPY rows: 5070, range: 2006-01-03 to 2026-02-27
